# 채점기 v3.1.1 재채점

기존 생성 결과를 다시 생성하지 않고 재채점합니다. 인용 내용과 마지막 줄 형식을 분리하고 문항별 실패 원인 후보를 기록합니다.

In [ ]:
# 1. 경로와 입력 결과 선택
import hashlib, json, os, subprocess
from pathlib import Path
import pandas as pd

ROOT = Path('/home/kongseok/sprint-public-procurement-rag-assistant')
os.chdir(ROOT)
SOURCE_RUN_DIR = ROOT / 'output/vlm_cached_v3_1_runs/20260910T022125Z'
assert SOURCE_RUN_DIR.exists(), f'원본 결과 폴더가 없습니다: {SOURCE_RUN_DIR}'
assert (SOURCE_RUN_DIR / 'inference.jsonl').exists(), 'inference.jsonl이 없습니다.'

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

repository_head = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=ROOT, text=True
).strip()
source_manifest_path = SOURCE_RUN_DIR / 'manifest.json'
source_manifest = json.loads(source_manifest_path.read_text(encoding='utf-8'))
print('원본 결과:', SOURCE_RUN_DIR)
print('현재 레포 커밋:', repository_head)
print('원본 생성 커밋:', source_manifest.get('generation_source_commit', '원본 manifest에 기록 없음'))

In [ ]:
# 2. Golden Set v3.1과 내장된 v3.1.1 진단 로직으로 재채점
import re
from collections import Counter
from src.data_processing.chunking import load_chunks
from src.evaluation.golden_set_v3_1 import GOLDEN_PATCH_VERSION, load_golden_set_v3_1
from src.evaluation.scoring_v3 import scorer_v3_1 as scorer

assert scorer.SCORER_VERSION == '3.1.0'
chunks = load_chunks()
corpus_doc_ids = {str(chunk.doc_id) for chunk in chunks}
golden_rows = load_golden_set_v3_1(corpus_doc_ids=corpus_doc_ids).to_dict('records')
predictions = scorer.read_jsonl(SOURCE_RUN_DIR / 'inference.jsonl')
assert len(golden_rows) == 79, len(golden_rows)
assert len(predictions) == 79, len(predictions)

details, summary = scorer.evaluate(golden_rows, predictions)
prediction_by_id = {str(row['id']): row for row in predictions}
golden_by_id = {str(row['id']): row for row in golden_rows}
citation_anywhere = re.compile(r'\[\s*근거\s*:\s*([^\]\r\n]+?)\s*\]', re.I)
citation_at_end = re.compile(r'\[\s*근거\s*:\s*([^\]\r\n]+?)\s*\]\s*$', re.I)

def add_v311_diagnostics(detail):
    text = str(detail.get('answer') or '')
    matches = list(citation_anywhere.finditer(text))
    citation_status = 'valid' if citation_at_end.search(text) else ('misplaced' if matches else 'missing')
    cited_raw = []
    if matches:
        cited_raw = [value.strip() for value in matches[-1].group(1).split(',') if value.strip()]
    gold = scorer._BASE.gold_citation_set(golden_by_id[str(detail['id'])])
    cited = scorer._BASE.normalized_document_set(cited_raw)
    retrieved = scorer._BASE.prediction_document_set(prediction_by_id[str(detail['id'])])
    matched = len(cited & gold)
    detail.update({
        'scorer_version': '3.1.1', 'scoring_schema_version': '3.1.1',
        'citation_status': citation_status,
        'citation_format_pass': citation_status == 'valid',
        'citation_content_available': bool(cited_raw),
        'cited_doc_ids': cited_raw,
        'citation_recall': matched / len(gold) if gold else None,
        'citation_precision': matched / len(cited) if cited else (0.0 if gold else None),
        'unsupported_citation_count': len(cited - retrieved) if retrieved else None,
    })
    reasons = []
    status = detail.get('response_status')
    if status in {'execution_error', 'empty_response'}:
        reasons.append(status)
    else:
        if detail.get('retrieval_recall') == 0: reasons.append('retrieval_failure')
        if detail.get('context_fact_coverage') == 0: reasons.append('context_fact_missing')
        if detail.get('abstention_match') is False: reasons.append('abstention_mismatch')
        if detail.get('list_f1') is not None and detail.get('list_f1') < 1: reasons.append('set_mismatch')
        if detail.get('lexical_fact_score') is not None and detail.get('lexical_fact_score') < 100: reasons.append('fact_incomplete_or_mismatch')
        if citation_status == 'missing': reasons.append('citation_missing')
        elif citation_status == 'misplaced': reasons.append('citation_misplaced')
        if (detail.get('unsupported_citation_count') or 0) > 0: reasons.append('unsupported_citation')
    detail['failure_reasons'] = reasons or ['none']

for detail in details:
    add_v311_diagnostics(detail)

def mean(field):
    values = [float(row[field]) for row in details if row.get(field) is not None]
    return sum(values) / len(values) if values else None

summary.update({
    'scorer_version': '3.1.1', 'scoring_schema_version': '3.1.1',
    'citation_format_pass_rate': mean('citation_format_pass'),
    'citation_content_available_rate': mean('citation_content_available'),
    'citation_recall': mean('citation_recall'), 'citation_precision': mean('citation_precision'),
    'citation_status_counts': dict(Counter(row['citation_status'] for row in details)),
    'failure_reason_counts': dict(Counter(reason for row in details for reason in row['failure_reasons'] if reason != 'none')),
})
OUT_DIR = ROOT / 'output/scoring_v3_1_1_runs' / SOURCE_RUN_DIR.name
OUT_DIR.mkdir(parents=True, exist_ok=True)
manifest = {
    'mode': 'rescore_existing_inference',
    'source_run_directory': str(SOURCE_RUN_DIR),
    'source_inference_sha256': sha256(SOURCE_RUN_DIR / 'inference.jsonl'),
    'source_manifest_sha256': sha256(source_manifest_path),
    'source_generation_commit': source_manifest.get('generation_source_commit'),
    'repository_head_at_rescore': repository_head,
    'scorer_version': '3.1.1',
    'golden_patch_version': GOLDEN_PATCH_VERSION,
}
summary.update(manifest)
scorer.write_jsonl(OUT_DIR / 'scored_details.jsonl', details)
pd.DataFrame(details).to_csv(OUT_DIR / 'scored_details.csv', index=False, encoding='utf-8-sig')
(OUT_DIR / 'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
(OUT_DIR / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print('결과 폴더:', OUT_DIR)

In [ ]:
# 3. 핵심 지표와 진단 확인
display(pd.DataFrame({
    '지표': ['End-to-End', '어휘 사실 점수', '인용 형식 통과율', '인용 내용 존재율', '인용 Recall', '인용 Precision'],
    '결과': [summary.get('end_to_end_score'), summary.get('average_lexical_fact_score'), summary.get('citation_format_pass_rate'), summary.get('citation_content_available_rate'), summary.get('citation_recall'), summary.get('citation_precision')],
}))
print('인용 상태:', summary.get('citation_status_counts'))
print('실패 원인 후보:', summary.get('failure_reason_counts'))
problem_rows = [row for row in details if row.get('failure_reasons') != ['none']]
display(pd.DataFrame(problem_rows)[['id', 'end_to_end_score', 'citation_status', 'failure_reasons']])